# Py Lab 4 — pandas and wrangling
**PUBHLT 0411 · Python for Public Health Data Analysis**

Name: 

Submit this notebook as a `.ipynb` file.

---

Work each exercise in the cell provided. Replace every blank (`____`) with working code, then run the cell to check it does what the task asks.

## Before you start — load the data

This lab uses:

- `pa_counties.csv`
- `pa_covid_monthly.csv`
- `pa_firearm_county_year.csv`
- `pa_maternal_infant.csv`
- `pa_overdose_county_year.csv`

Run the setup cell below **once**, at the start of the session. It looks for the files in your Drive folder and falls back to an upload prompt if it cannot find them.

If you have not made the Drive folder yet: create a folder called `pubhlt0411` in the top level of your Google Drive and put the CSVs in it. That is the folder set up in Module 0.

In [ ]:
# Setup — run this cell once, before anything else.
#
# Downloads the lab's CSVs from the course site into a local `data/` folder.
# Nothing to upload and no Google Drive needed. Re-run it if the session restarts.
import os
import urllib.request

COURSE_DATA = "https://soumikp.github.io/pubhlt0411-python/data/"

FILES = [
    "pa_counties.csv",
    "pa_covid_monthly.csv",
    "pa_firearm_county_year.csv",
    "pa_maternal_infant.csv",
    "pa_overdose_county_year.csv"
]

os.makedirs("data", exist_ok=True)

for name in FILES:
    target = os.path.join("data", name)
    if not os.path.exists(target):
        urllib.request.urlretrieve(COURSE_DATA + name, target)

print("Data ready:", ", ".join(FILES))

---

## Exercise 1 — Read and inspect

**Worked example.** `pd.read_csv()` reads a whole table in one call. `.shape` gives rows and
columns; `.columns` lists the names.

In [ ]:
import pandas as pd

counties = pd.read_csv("data/pa_counties.csv")

print(counties.shape)
print(counties.columns.tolist())
print(counties.head(2))

**Your turn.** Read the maternal and infant health file and report its dimensions.

In [ ]:
import pandas as pd

births = pd.____("data/pa_maternal_infant.csv")

n_rows = births.shape[0]
n_cols = births.shape[1]

print(f"Rows: {n_rows}   Columns: {n_cols}")
print(births.columns.tolist())

---

## Exercise 2 — Find the missing values

**Worked example.** `.isna()` marks each cell as missing or not; `.sum()` counts the missing
ones per column, because `True` equals 1.

In [ ]:
import pandas as pd

counties = pd.read_csv("data/pa_counties.csv")

print(counties.isna().sum())

**Your turn.** Count the missing values in every column of the births file, then pull out the
count for infant mortality.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

missing_per_column = births.____().sum()
n_suppressed = missing_per_column["infant_mortality_per_1000"]

print(missing_per_column)
print(f"\nCounties with no published infant mortality rate: {n_suppressed}")

---

## Exercise 3 — Select a column

**Worked example.** Square brackets with a column name return a **Series** — one column plus
its index. `.count()` counts present values; `.size` counts rows.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

poverty = births["children_in_poverty_pct"]

print(type(poverty))
print(f"Mean: {poverty.mean():.2f}   Present: {poverty.count()} of {poverty.size}")

**Your turn.** Select the low birthweight column and summarise it. Every county reports this
one.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

low_birthweight = births[____]

mean_pct = low_birthweight.mean()
highest = low_birthweight.max()
n_present = low_birthweight.count()

print(f"Mean:    {mean_pct:.2f}%")
print(f"Highest: {highest}%")
print(f"Reporting: {n_present} counties")

---

## Exercise 4 — Filter rows

**Worked example.** A comparison on a column produces a boolean mask. Passing it to `[]`
keeps the rows where it is `True`, and **every column comes along**.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

poor = births[births["children_in_poverty_pct"] > 20]

print(poor.shape)
print(poor[["county", "children_in_poverty_pct"]].head(3))

**Your turn.** Select the counties where more than 9% of births are low birthweight.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

high_lbw = births[births["low_birthweight_pct"] ____ 9]

n_counties = len(high_lbw)

print(f"Counties above 9%: {n_counties}")
print(high_lbw[["county", "low_birthweight_pct"]].to_string(index=False))

---

## Exercise 5 — Combine two conditions

**Worked example.** `&` means and, `|` means or. **Each comparison needs its own
parentheses**, because `&` binds more tightly than `>`.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

both = births[(births["children_in_poverty_pct"] > 20) & (births["low_birthweight_pct"] > 8)]

print(both[["county", "children_in_poverty_pct", "low_birthweight_pct"]].to_string(index=False))

**Your turn.** Find the counties where more than 6% of children are uninsured **and** more
than 15% live in poverty. Supply both missing operators.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

strained = births[
    (births["uninsured_children_pct"] > 6) ____ (births["children_in_poverty_pct"] ____ 15)
]

n_strained = len(strained)

print(f"Counties meeting both conditions: {n_strained}")
print(strained[["county", "uninsured_children_pct", "children_in_poverty_pct"]].to_string(index=False))

---

## Exercise 6 — Derive a column

**Worked example.** `.assign()` returns a new DataFrame with a column added. Inside it,
`lambda d:` refers to the table as it exists at that point in the chain.

In [ ]:
import pandas as pd

overdose = pd.read_csv("data/pa_overdose_county_year.csv")

checked = overdose[overdose["year"] == 2023].assign(
    rate_check=lambda d: d["overdose_deaths"] / d["population"] * 100_000
)

print(checked[["county", "rate_per_100k", "rate_check"]].head(3).round(2))

**Your turn.** Compute the 2022 firearm death rate per 100,000 and confirm it reproduces the
published column.

In [ ]:
import pandas as pd

firearm = pd.read_csv("data/pa_firearm_county_year.csv")

f2022 = firearm[firearm["year"] == 2022].assign(
    my_rate=lambda d: d["firearm_deaths"] / d["population"] * ______
)

allegheny = f2022[f2022["county"] == "Allegheny"]

print(allegheny[["county", "firearm_deaths", "population", "rate_per_100k", "my_rate"]].round(2).to_string(index=False))

---

## Exercise 7 — Sort and rank

**Worked example.** `.sort_values()` orders rows and carries every column with them. Missing
values sort to the end regardless of direction.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

print(births.sort_values("children_in_poverty_pct", ascending=False)[
    ["county", "children_in_poverty_pct"]
].head(3).to_string(index=False))

**Your turn.** Rank the counties by teen birth rate, highest first, and show the top five.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

ranked = births.sort_values("teen_births_per_1000", ascending=____)

top5 = ranked[["county", "teen_births_per_1000"]].head(5)

print(top5.to_string(index=False))
print(f"\nHighest: {ranked.iloc[0]['county']}")

---

## Exercise 8 — Group and aggregate

**Worked example.** `.groupby()` splits rows into groups, applies an aggregation to each, and
combines the results. `.agg()` takes several functions at once.

In [ ]:
import pandas as pd

overdose = pd.read_csv("data/pa_overdose_county_year.csv")

print(overdose.groupby("year")["rate_per_100k"].agg(["count", "mean"]).round(2))

**Your turn.** Summarise firearm deaths by year: how many counties reported, and the mean
rate among them.

In [ ]:
import pandas as pd

firearm = pd.read_csv("data/pa_firearm_county_year.csv")

by_year = firearm.____("year")["rate_per_100k"].agg(["count", "mean"]).round(2)

print(by_year)
print(f"\nYears covered: {len(by_year)}")

---

## Exercise 9 — Join two tables

**Worked example.** `.merge()` joins on a shared key. `how="left"` keeps every row of the
left table and fills unmatched columns with `NaN`.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")
counties = pd.read_csv("data/pa_counties.csv")

joined = births.merge(counties, on="county", how="left")

print(f"{births.shape} + {counties.shape} -> {joined.shape}")
print(joined[["county", "low_birthweight_pct", "region", "metro"]].head(2))

**Your turn.** The overdose file has no `region` column. Attach one, keeping every overdose
row.

In [ ]:
import pandas as pd

overdose = pd.read_csv("data/pa_overdose_county_year.csv")
counties = pd.read_csv("data/pa_counties.csv")

o2023 = overdose[overdose["year"] == 2023]

joined = o2023.merge(counties, ____="county", how="left")

rows_before = len(o2023)
rows_after = len(joined)

print(f"Before: {rows_before} rows   After: {rows_after} rows")
print(f"Regions attached: {joined['region'].nunique()}")
print(joined[["county", "rate_per_100k", "region", "metro"]].head(3).to_string(index=False))

---

## Exercise 10 — Group by a joined column

**Worked example.** Once `metro` is attached, it can be grouped on. Neither grouping variable
exists in any outcome file, so **the join is what makes the comparison possible**.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")
counties = pd.read_csv("data/pa_counties.csv")

joined = births.merge(counties, on="county", how="left")

print(joined.groupby("metro")["median_household_income"].agg(["size", "median"]).round(0))

**Your turn.** Compare mean low birthweight across the six regions, and report how many
counties each region contains.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")
counties = pd.read_csv("data/pa_counties.csv")

joined = births.merge(counties, on="county", how="left")

by_region = joined.groupby("____")["low_birthweight_pct"].agg(["size", "mean"]).round(2)

print(by_region.sort_values("mean", ascending=False))

---

## Exercise 11 — Cross-tabulate

**Worked example.** `pd.crosstab()` counts one categorical against another. `.isna()` on a
column produces exactly such a categorical: suppressed or not.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")
counties = pd.read_csv("data/pa_counties.csv")

joined = births.merge(counties, on="county", how="left")
joined["suppressed"] = joined["infant_mortality_per_1000"].isna()

print(pd.crosstab(joined["metro"], joined["suppressed"]))

**Your turn.** Firearm deaths are suppressed too. Cross-tabulate 2023 suppression against
metro status.

In [ ]:
import pandas as pd

firearm = pd.read_csv("data/pa_firearm_county_year.csv")
counties = pd.read_csv("data/pa_counties.csv")

f2023 = firearm[firearm["year"] == 2023].merge(counties, on="county", how="left")
f2023["suppressed"] = f2023["firearm_deaths"].isna()

table = pd.____(f2023["metro"], f2023["suppressed"])

print(table)
print(f"\nTotal suppressed: {f2023['suppressed'].sum()} of {len(f2023)}")

---

## Exercise 12 — The suppression bias

**Worked example.** A grouped rate divides summed deaths by summed population. `.sum()` skips
`NaN` — but the population denominator still counts every county.

In [ ]:
import pandas as pd

overdose = pd.read_csv("data/pa_overdose_county_year.csv")
counties = pd.read_csv("data/pa_counties.csv")

o = overdose[overdose["year"] == 2023].merge(counties, on="county", how="left")
nw = o[o["region"] == "Northwest"]

print(f"Counties: {len(nw)}, of which suppressed: {nw['overdose_deaths'].isna().sum()}")
print(f"Naive rate: {nw['overdose_deaths'].sum() / nw['population_2023'].sum() * 100_000:.1f}")

**Your turn.** Recompute the Northwest rate over the counties that actually reported, and
measure the bias. Drop the suppressed rows before summing.

In [ ]:
import pandas as pd

overdose = pd.read_csv("data/pa_overdose_county_year.csv")
counties = pd.read_csv("data/pa_counties.csv")

o = overdose[overdose["year"] == 2023].merge(counties, on="county", how="left")
nw = o[o["region"] == "Northwest"]

naive = nw["overdose_deaths"].sum() / nw["population_2023"].sum() * 100_000

reported = nw.____(subset=["overdose_deaths"])
correct = reported["overdose_deaths"].sum() / reported["population_2023"].sum() * 100_000

print(f"Naive:   {naive:.1f} per 100,000  (denominator: {len(nw)} counties)")
print(f"Correct: {correct:.1f} per 100,000  (denominator: {len(reported)} counties)")
print(f"Bias:    {naive - correct:.1f}")

---

## Exercise 13 — Dates

**Worked example.** `pd.to_datetime()` converts a text column to dates, which then sort
chronologically and expose `.dt.year` and `.dt.month`.

In [ ]:
import pandas as pd

covid = pd.read_csv("data/pa_covid_monthly.csv")
covid["month"] = pd.to_datetime(covid["month"])

print(covid["month"].dtype)
print(f"{covid['month'].min().date()} to {covid['month'].max().date()}")

**Your turn.** Total COVID deaths by month across the state, then find the worst month.

In [ ]:
import pandas as pd

covid = pd.read_csv("data/pa_covid_monthly.csv")
covid["month"] = pd.____(covid["month"])

by_month = covid.groupby("month")["covid_deaths"].sum()

peak_month = by_month.idxmax()
peak_deaths = by_month.max()

print(f"Months covered: {by_month.size}")
print(f"Worst month:    {peak_month.date()}  ({peak_deaths} deaths)")
print()
print(covid.groupby(covid["month"].dt.year)["covid_deaths"].sum())

---

## Exercise 14 — Repair a broken key

**Your turn.** Total COVID deaths by month across the state, then find the worst month.

```{pyodide}
#| exercise: ex13_dates
import pandas as pd

covid = pd.read_csv("data/pa_covid_monthly.csv")
covid["month"] = pd.____(covid["month"])

by_month = covid.groupby("month")["covid_deaths"].sum()

peak_month = by_month.idxmax()
peak_deaths = by_month.max()

print(f"Months covered: {by_month.size}")
print(f"Worst month:    {peak_month.date()}  ({peak_deaths} deaths)")
print()
print(covid.groupby(covid["month"].dt.year)["covid_deaths"].sum())
```

```{pyodide}
#| exercise: ex13_dates
#| hint: true
# The blank is to_datetime — a function on pd, not a method on the column.
# .idxmax() returns the INDEX LABEL of the largest value, where .max()
# returns the value itself. Here the index is the month, so idxmax gives a
# date. This is the same value-versus-position distinction as NumPy's
# .argmax() in Module 3.
# The peak is January 2022 — the Omicron wave — at 4,294 deaths.
```

```{pyodide}
#| exercise: ex13_dates
#| solution: true
import pandas as pd

covid = pd.read_csv("data/pa_covid_monthly.csv")
covid["month"] = pd.to_datetime(covid["month"])

by_month = covid.groupby("month")["covid_deaths"].sum()

peak_month = by_month.idxmax()
peak_deaths = by_month.max()

print(f"Months covered: {by_month.size}")
print(f"Worst month:    {peak_month.date()}  ({peak_deaths} deaths)")
print()
print(covid.groupby(covid["month"].dt.year)["covid_deaths"].sum())
```

```{pyodide}
#| exercise: ex13_dates
#| check: true
import pandas as pd
assert pd.api.types.is_datetime64_any_dtype(covid["month"]), "The month column must be datetime."
assert by_month.size == 49, "The file covers 49 months, March 2020 to March 2024."
assert str(peak_month.date()) == "2022-01-01", "January 2022 is the Omicron peak."
assert peak_deaths == 4294, "The peak month recorded 4,294 deaths statewide."
```

# Exercise 14 — Repair a broken key 

## Exercise 14 — Repair a broken key 

**This join is broken.** Run it, read the output, and repair it.

County Health Rankings publishes one county as **"Mc Kean"**; the Census file writes
**"McKean"**. The two strings differ by one space, so the row does not match.

In [ ]:
import pandas as pd

counties = pd.read_csv("data/pa_counties.csv")

external = pd.DataFrame({
    "county": ["Mc Kean", "Potter", "Tioga"],
    "clinics": [3, 2, 4],
})

fixed = external.assign(
    county=lambda d: d["county"].str.replace(" ", "", regex=False)
)

joined = fixed.merge(counties[["county", "population_2023"]], on="county", how="left")
n_unmatched = joined["population_2023"].isna().sum()

print(joined)
print(f"\nUnmatched rows: {n_unmatched}")

---

## Exercise 15 — Rename and drop

**Worked example.** `.rename(columns={old: new})` renames by mapping; `.drop(columns=[...])`
removes columns. Both return a new DataFrame.

In [ ]:
import pandas as pd

counties = pd.read_csv("data/pa_counties.csv")

trimmed = counties.rename(columns={"population_2023": "population"}).drop(columns=["fips"])

print(trimmed.columns.tolist())

**Your turn.** Shorten two column names and remove the two columns this analysis does not use.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")

tidy = (
    births
    .rename(____={
        "infant_mortality_per_1000": "infant_mortality",
        "children_in_poverty_pct": "child_poverty",
    })
    .drop(columns=["teen_births_per_1000", "uninsured_children_pct"])
)

print(tidy.columns.tolist())
print(f"Columns: {births.shape[1]} -> {tidy.shape[1]}")

---

## Exercise 16 — Chain the pipeline

**Worked example.** Each method returns a DataFrame, so the next attaches to it. The outer
parentheses let the chain break across lines.

In [ ]:
import pandas as pd

births = pd.read_csv("data/pa_maternal_infant.csv")
counties = pd.read_csv("data/pa_counties.csv")

print(
    births
    .merge(counties, on="county", how="left")
    .groupby("metro")
    .agg(counties=("county", "size"), mean_lbw=("low_birthweight_pct", "mean"))
    .round(2)
)

**Your turn.** Build the full pipeline: take 2023 overdose data, attach the region, drop the
suppressed counties, and summarise by region.

In [ ]:
import pandas as pd

overdose = pd.read_csv("data/pa_overdose_county_year.csv")
counties = pd.read_csv("data/pa_counties.csv")

summary = (
    overdose
    .query("year == 2023")
    .merge(counties, on="county", how="left")
    .dropna(subset=["overdose_deaths"])
    .____("region")
    .agg(reporting=("county", "size"), mean_rate=("rate_per_100k", "mean"))
    .sort_values("mean_rate", ascending=False)
    .round(1)
)

print(summary)